In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import HTML

from poc.entities import DEFAULT_ROBOT_SPEED_MPS
from poc.opponent_policy import build_opponent_policy
from poc.planner import UtilityPlanner
from poc.rl_checkpoint import load_checkpoint
from poc.rl_config import selfplay_config_from_dict
from poc.rl_model import load_compatible_state_dict
from poc.rl_selfplay import TorchPolicySelector, build_model
from poc.scenarios import build_scenario
from poc.simulator import Simulator, save_result
from poc.visualize import animate_match_overview, plot_match_overview, save_animation_media

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUN_NAME = "ppo_v19"
RUN_DIR = ROOT / "runs" / RUN_NAME
METRICS_PATH = RUN_DIR / "metrics.jsonl"
CHECKPOINT_PATH = RUN_DIR / "latest.pt"

SCENARIO_NAME = "baseline"
OPPONENT_NAME = "yellow_side_fixed_sequence"
MATCH_SEED = 12

OUR_SPEED_MULTIPLIER = 1.0
ENEMY_SPEED_MULTIPLIER = 1.0
OUR_SPEED_MPS = DEFAULT_ROBOT_SPEED_MPS * OUR_SPEED_MULTIPLIER
ENEMY_SPEED_MPS = DEFAULT_ROBOT_SPEED_MPS * ENEMY_SPEED_MULTIPLIER

SAVE_ANIMATION = False
ANIMATION_FORMAT = "mp4"  # "gif", "html", or "mp4"
ANIMATION_STEM = ROOT / "runs" / f"{RUN_NAME}_{CHECKPOINT_PATH.stem}_animation"
ANIMATION_FPS = 4


def load_metrics_df(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        row = json.loads(line)
        rollout = row.get("rollout", {})
        row["rollout_steps"] = rollout.get("steps")
        row["rollout_episodes"] = rollout.get("episodes")
        row["rollout_policy_loss"] = rollout.get("policy_loss")
        row["rollout_value_loss"] = rollout.get("value_loss")
        row["rollout_entropy"] = rollout.get("entropy")
        row["rollout_total_loss"] = rollout.get("total_loss")
        row["eval_present"] = bool(row.get("evaluation"))
        if not row["eval_present"]:
            for key in (
                "overall_eval_winrate",
                "overall_eval_score_diff",
                "overall_eval_p10_score_diff",
                "overall_eval_min_score_diff",
                "scripted_eval_winrate",
                "scripted_eval_score_diff",
                "scripted_eval_p10_score_diff",
                "scripted_eval_min_score_diff",
                "randomized_eval_winrate",
                "randomized_eval_score_diff",
                "randomized_eval_p10_score_diff",
                "randomized_eval_min_score_diff",
                "self_play_eval_winrate",
                "self_play_eval_score_diff",
                "self_play_eval_p10_score_diff",
                "self_play_eval_min_score_diff",
                "robust_eval_winrate",
                "robust_eval_score_diff",
                "robust_eval_p10_score_diff",
                "robust_eval_min_score_diff",
            ):
                row[key] = None
        rows.append(row)
    return pd.DataFrame(rows)


metrics_df = load_metrics_df(METRICS_PATH)
metrics_df[
    [
        "update",
        "rollout_total_loss",
        "rollout_entropy",
        "scripted_eval_winrate",
        "randomized_eval_winrate",
        "self_play_eval_winrate",
        "robust_eval_winrate",
        "scripted_eval_score_diff",
        "randomized_eval_score_diff",
        "robust_eval_score_diff",
    ]
].tail(10)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharex='col')

axes[0, 0].plot(metrics_df["update"], metrics_df["rollout_total_loss"], label="total_loss")
axes[0, 0].plot(metrics_df["update"], metrics_df["rollout_value_loss"], label="value_loss", alpha=0.7)
axes[0, 0].set_title("Rollout Losses")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(metrics_df["update"], metrics_df["rollout_entropy"], label="entropy", color="tab:orange")
axes[0, 1].set_title("Rollout Entropy")
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(metrics_df["update"], metrics_df["rollout_steps"], label="steps", color="tab:green")
axes[0, 2].plot(metrics_df["update"], metrics_df["rollout_episodes"], label="episodes", color="tab:red", alpha=0.7)
axes[0, 2].set_title("Rollout Size")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

eval_df = metrics_df[metrics_df["eval_present"]].copy()

axes[1, 0].plot(eval_df["update"], eval_df["scripted_eval_winrate"], marker="o", label="scripted")
axes[1, 0].plot(eval_df["update"], eval_df["randomized_eval_winrate"], marker="o", label="randomized")
axes[1, 0].plot(eval_df["update"], eval_df["self_play_eval_winrate"], marker="o", label="self_play")
axes[1, 0].plot(eval_df["update"], eval_df["robust_eval_winrate"], marker="o", label="robust", linewidth=2.0)
axes[1, 0].plot(eval_df["update"], eval_df["overall_eval_winrate"], marker="o", label="overall", alpha=0.7)
axes[1, 0].set_title("Eval Winrate")
axes[1, 0].set_ylim(-0.05, 1.05)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(eval_df["update"], eval_df["scripted_eval_score_diff"], marker="o", label="mean")
axes[1, 1].plot(eval_df["update"], eval_df["scripted_eval_p10_score_diff"], marker="o", label="p10")
axes[1, 1].plot(eval_df["update"], eval_df["scripted_eval_min_score_diff"], marker="o", label="min")
axes[1, 1].set_title("Scripted Eval Score Diff")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(eval_df["update"], eval_df["randomized_eval_score_diff"], marker="o", label="randomized mean")
axes[1, 2].plot(eval_df["update"], eval_df["self_play_eval_score_diff"], marker="o", label="self_play mean")
axes[1, 2].plot(eval_df["update"], eval_df["robust_eval_score_diff"], marker="o", label="robust mean", linewidth=2.0)
axes[1, 2].plot(eval_df["update"], eval_df["robust_eval_p10_score_diff"], marker="o", label="robust p10", linestyle="--")
axes[1, 2].set_title("Robust Eval Score Diff")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

for ax in axes[-1, :]:
    ax.set_xlabel("update")

plt.tight_layout()
fig


In [ ]:
payload = load_checkpoint(CHECKPOINT_PATH, map_location="cpu")
config = selfplay_config_from_dict(dict(payload["config"]))

model = build_model(config)
load_compatible_state_dict(model, payload["model_state"])
model.eval()

selector = TorchPolicySelector(
    model=model,
    device="cpu",
    greedy=True,
    name=f"{RUN_NAME}_{CHECKPOINT_PATH.stem}",
)

scenario = build_scenario(
    SCENARIO_NAME,
    seed=MATCH_SEED,
    our_side=config.side,
    opponent_policy_name=OPPONENT_NAME,
    our_robot_speed=OUR_SPEED_MPS,
    enemy_robot_speed=ENEMY_SPEED_MPS,
)

sim = Simulator(
    state=scenario.game_state,
    scenario_name=scenario.name,
    opponent_policy=build_opponent_policy(OPPONENT_NAME),
    planner=UtilityPlanner(),
    dt=config.dt,
    action_selectors={config.side: selector},
)

result = sim.run()
match_output = ROOT / "runs" / f"{RUN_NAME}_{CHECKPOINT_PATH.stem}_vs_{OPPONENT_NAME}_seed{MATCH_SEED}.json"
save_result(result, match_output)
result.summary


In [ ]:
fig = plot_match_overview(result)
fig


In [ ]:
anim = animate_match_overview(result, interval=80, frame_stride=2)

if SAVE_ANIMATION:
    output_path = ANIMATION_STEM.with_suffix(f".{ANIMATION_FORMAT}")
    saved_path = save_animation_media(anim, output_path, fps=ANIMATION_FPS)
    print(f"Saved animation to {saved_path}")

HTML(anim.to_jshtml())
